# Importing Libraries

In [7]:
# Install necessary libraries (if not already installed)
!pip install tensorflow emoji pyspellchecker wordcloud plotly


In [9]:
# Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import nltk
from wordcloud import WordCloud, STOPWORDS
import plotly.express as px
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from spellchecker import SpellChecker
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, GlobalAveragePooling1D, TextVectorization # Import TextVectorization directly
from sklearn.model_selection import train_test_split

# Download NLTK data
nltk.download('stopwords')
nltk.download('wordnet')

# Load Dataset
twitter_data = pd.read_csv('Twitter_Data.csv')
reddit_data = pd.read_csv('Reddit_Data.csv')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [10]:

# Rename Columns
twitter_data.columns = ['messages', 'labels']
reddit_data.columns = ['messages', 'labels']


In [11]:

# Merge Datasets
data = pd.concat([twitter_data, reddit_data], ignore_index=True)


In [12]:

# Drop Missing Values
data = data.dropna()


In [13]:

# Shuffle Dataset
data = data.sample(frac=1)


In [14]:

# Data Cleaning Functions
def clean_text(text):
    text = str(text).lower()
    text = re.sub('https?:\/\/\S*|www\.\S+', 'URL', text)  # Replace URLs
    text = re.sub('<.*?>', '', text)  # Remove HTML
    text = re.sub('@\S*', 'user', text, flags=re.IGNORECASE)  # Replace mentions
    text = re.sub('[0-9]+', 'NUMBER', text)  # Replace numbers
    text = re.sub('\w*\d+\w*', '', text)  # Remove alphanumeric words
    text = ''.join([char for char in text if char not in string.punctuation])  # Remove punctuations
    text = ' '.join([word for word in text.split() if word not in stopwords.words('english')])  # Remove stopwords
    lemmatizer = WordNetLemmatizer()
    text = ' '.join([lemmatizer.lemmatize(word) for word in text.split()])  # Lemmatization
    return text


In [15]:

# Apply Cleaning
data['cleaned_messages'] = data['messages'].apply(clean_text)


In [16]:

# Split Dataset
X = data['cleaned_messages']
y = data['labels']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Shift Labels to [0, 2] Range (Moved here)
y_train = y_train + 1
y_test = y_test + 1


In [17]:

# TensorFlow Text Vectorization
vectorizer = TextVectorization(max_tokens=10000, output_sequence_length=100)
vectorizer.adapt(X_train)



In [18]:
# Vectorize Text Data
X_train = vectorizer(X_train).numpy()
X_test = vectorizer(X_test).numpy()


In [19]:
# Shift Labels to [0, 2] Range
y_train = y_train + 1
y_test = y_test + 1

# Convert Labels to TensorFlow Format
y_train = tf.convert_to_tensor(y_train.values)
y_test = tf.convert_to_tensor(y_test.values)


In [20]:

# Build TensorFlow Model
model = Sequential([
    Embedding(input_dim=10000, output_dim=128),
    GlobalAveragePooling1D(),
    Dense(64, activation='relu'),
    Dense(3, activation='softmax')  # 3 classes: Positive, Neutral, Negative
])


In [23]:

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])


history = model.fit(X_train, y_train, epochs=10, validation_data=(X_test, y_test), batch_size=32)

# Evaluate Model
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {accuracy * 100:.2f}%")

# Define Label Mapping
label_mapping = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}


Epoch 1/10
5005/5005 ━━━━━━━━━━━━━━━━━━━━ 15s 3ms/step - accuracy: 0.3407 - loss: nan - val_accuracy: 0.3401 - val_loss: nan
Epoch 2/10
5005/5005 ━━━━━━━━━━━━━━━━━━━━ 18s 2ms/step - accuracy: 0.3399 - loss: nan - val_accuracy: 0.3401 - val_loss: nan
Epoch 3/10
5005/5005 ━━━━━━━━━━━━━━━━━━━━ 21s 2ms/step - accuracy: 0.3418 - loss: nan - val_accuracy: 0.3401 - val_loss: nan
Epoch 4/10
5005/5005 ━━━━━━━━━━━━━━━━━━━━ 20s 2ms/step - accuracy: 0.3412 - loss: nan - val_accuracy: 0.3401 - val_loss: nan
Epoch 5/10
5005/5005 ━━━━━━━━━━━━━━━━━━━━ 22s 3ms/step - accuracy: 0.3397 - loss: nan - val_accuracy: 0.3401 - val_loss: nan
Epoch 6/10
5005/5005 ━━━━━━━━━━━━━━━━━━━━ 19s 2ms/step - accuracy: 0.3405 - loss: nan - val_accuracy: 0.3401 - val_loss: nan
Epoch 7/10
5005/5005 ━━━━━━━━━━━━━━━━━━━━ 20s 2ms/step - accuracy: 0.3413 - loss: nan - val_accuracy: 0.3401 - val_loss: nan
Epoch 8/10
5005/5005 ━━━━━━━━━━━━━━━━━━━━ 13s 2ms/step - accuracy: 0.3419 - loss: nan - val_accuracy: 0.3401 - val_loss: nan


In [24]:

# User Input Prediction
def predict_sentiment(input_text):
    # Clean and preprocess input text
    cleaned_text = clean_text(input_text)
    vectorized_text = vectorizer([cleaned_text])  # Vectorize text
    prediction = model.predict(vectorized_text)  # Get model prediction
    sentiment = np.argmax(prediction)  # Get the predicted class index
    confidence = np.max(prediction)  # Get the confidence score
    sentiment_label = label_mapping[sentiment]
    return sentiment_label, confidence


In [ ]:

# Loop to Catch User Input
while True:
    user_input = input("Enter your message (or type 'exit' to quit): ")
    if user_input.lower() == 'exit':
        print("Goodbye!")
        break
    sentiment, confidence = predict_sentiment(user_input)
    print(f"Sentiment: {sentiment} (Confidence: {confidence * 100:.2f}%)")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step
Sentiment: Negative (Confidence: nan%)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
Sentiment: Negative (Confidence: nan%)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
Sentiment: Negative (Confidence: nan%)
